In [1]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("PartitioningExample") \
    .master("local[*]") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/24 07:50:08 WARN Utils: Your hostname, codespaces-85d69f, resolves to a loopback address: 127.0.0.1; using 10.0.4.172 instead (on interface eth0)
26/06/24 07:50:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/24 07:50:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print("Generating 5,000,000 records...")
df = spark.range(0, 5000000)

Generating 5,000,000 records...


26/06/24 07:51:54 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

In [4]:
initial_partitions = df.rdd.getNumPartitions()
print(f"Initial number of partitions: {initial_partitions}")
print(f"Initial row count: {df.count()}")
print("-" * 40)

Initial number of partitions: 2


Initial row count: 5000000
----------------------------------------


In [5]:
print("Increasing partitions to 12 using repartition()...")
df_repartitioned = df.repartition(12)
new_partitions = df_repartitioned.rdd.getNumPartitions()
print(f"Number of partitions after repartition(): {new_partitions}")
print("-" * 40)

Increasing partitions to 12 using repartition()...


Number of partitions after repartition(): 12
----------------------------------------


In [6]:
print("Reducing partitions to 3 using coalesce()...")
df_coalesced = df_repartitioned.coalesce(3)
final_partitions = df_coalesced.rdd.getNumPartitions()
print(f"Final number of partitions after coalesce(): {final_partitions}")
print("-" * 40)

Reducing partitions to 3 using coalesce()...


Final number of partitions after coalesce(): 3
----------------------------------------


In [7]:
print(f"Final row count: {df_coalesced.count()}")
spark.stop()

Final row count: 5000000
